0#%% md
# Dimensioning a village battery for [Solbyn](https://solbyn.org/)

Knowns:
* AC cables connecting (in decreasing order of energy):
  1. A windmill far outside the village, outside the village AC voltage transformer.
  2. Northern and southern half-village with 25 households. All with hot-water accumulators. Many with wood stoves. Some with PEV panels. One with a solar heat panel.
  3. Eastern installation: PEV panels, solar heat panels, a large hot-water accumulator, a washing facility, a kindergarten, a cafeteria, a guest apartment.
  4. Western installation: PEV panels, 14 kWh electric battery and 20 EVs.
  5. Electric outdoor lighting.
* Data on power transfers to one household.
* Data on power transfers to and from the village except the windmill and households.

Suggestions:
* Increase electric battery to 36 kWh.
* Load-balance all households within each 10 buildings.
* Upgrade EV chargers to support vehicle-to-grid (VTG/VTX).
* Load-balance everything except the windmill.
* Load-balance everything including the windmill.

General questions:
* How many years can we expect before pay-off of a battery capacity increase to 36 kWh?
* Would the battery investment make sense with VTG?
* Would the battery investment make sense with everything except the windmill balanced?
* Would the battery investment make sense with everything including the windmill balanced?

First task:
* Is a flat electricity energy daily consumption a relevant approximation for dimensioning a battery that may later be used by all 50 households too?
* Use https://github.com/Arcascope/circadian to calculate daily and nightly energy transfers!


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as lines
import matplotlib_inline
%matplotlib inline

In [2]:
import pandas as pd

In [3]:
# Import with closest latitude/longitude time zone for Solbyn as default
TZ = 'Europe/Copenhagen'
def read_european_table(dataset: str, datum='Datum', tz=TZ) -> pd.DataFrame:
    df = pd.read_table(dataset, sep=';', decimal=',', index_col=datum, parse_dates=[datum], na_values=["-", "—"])
    df.index = df.index.tz_localize(tz, nonexistent="shift_forward", ambiguous="NaT")
    return df


In [4]:
systems = read_european_table('data/El - Sandbyvägen 196, Dalby.csv')
household = read_european_table('data/El - Sandbyvägen 158, Dalby.csv')

In [5]:
systems.head()

,Produktion,El kWh,Utomhustemperatur,kWh - 2023-01-01 - 2023-12-31,T(°C) - 2023-01-01 - 2023-12-31
Datum,,,,,
2024-01-01 00:00:00+01:00,0.0,6.17,4.2,8.79,5.7
2024-01-01 01:00:00+01:00,0.0,6.12,4.1,8.71,6.0
2024-01-01 02:00:00+01:00,0.0,5.61,3.9,7.95,6.6
2024-01-01 03:00:00+01:00,0.0,5.86,3.8,5.28,6.5
2024-01-01 04:00:00+01:00,0.0,5.91,3.8,5.84,5.9


In [6]:
household.head()

,El kWh,Utomhustemperatur,kWh - 2023-01-01 - 2023-12-31,T(°C) - 2023-01-01 - 2023-12-31
Datum,,,,
2024-01-01 00:00:00+01:00,1.412,4.2,1.148,5.7
2024-01-01 01:00:00+01:00,1.219,4.1,0.856,6.0
2024-01-01 02:00:00+01:00,2.172,3.9,0.891,6.6
2024-01-01 03:00:00+01:00,2.133,3.8,1.238,6.5
2024-01-01 04:00:00+01:00,1.179,3.8,1.187,5.9


In [7]:
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="village_battery")
location = geolocator.geocode("Sandbyvägen 196, Dalby, Sweden")
location

Location(196, Sandbyvägen, Dalby, Lunds kommun, Skåne län, 247 51, Sverige, (55.6731594, 13.3532784, 0.0))

In [8]:
import requests

latitude, longitude, _ = location.point
OPENTOPO = "https://api.opentopodata.org/v1/"
elevation = float(requests.get(f"{OPENTOPO}aster30m?locations={latitude},{longitude}").json()['results'][0]['elevation'])
elevation

86.0

In [9]:
from pvlib import location, solarposition, irradiance

site = location.Location(latitude=latitude, longitude=longitude, tz=TZ, altitude=elevation, name="Solbyn")
site

Location: 
  name: Solbyn
  latitude: 55.6731594
  longitude: 13.3532784
  altitude: 86.0
  tz: Europe/Copenhagen

In [10]:
datum = systems.index
clearsky = site.get_clearsky(datum, model="ineichen")
clearsky

,ghi,dni,dhi
Datum,,,
2024-01-01 00:00:00+01:00,0.0,0.0,0.0
2024-01-01 01:00:00+01:00,0.0,0.0,0.0
2024-01-01 02:00:00+01:00,0.0,0.0,0.0
2024-01-01 03:00:00+01:00,0.0,0.0,0.0
2024-01-01 04:00:00+01:00,0.0,0.0,0.0
...,...,...,...
2024-12-31 19:00:00+01:00,0.0,0.0,0.0
2024-12-31 20:00:00+01:00,0.0,0.0,0.0
2024-12-31 21:00:00+01:00,0.0,0.0,0.0


In [11]:
print('>>> systems.head()\n', systems.head())
print('>>> site\n', site)
print('>>> clearsky\n', clearsky)

>>> systems.head()
                            Produktion  El kWh  Utomhustemperatur  \
Datum                                                              
2024-01-01 00:00:00+01:00         0.0    6.17                4.2   
2024-01-01 01:00:00+01:00         0.0    6.12                4.1   
2024-01-01 02:00:00+01:00         0.0    5.61                3.9   
2024-01-01 03:00:00+01:00         0.0    5.86                3.8   
2024-01-01 04:00:00+01:00         0.0    5.91                3.8   

                           kWh - 2023-01-01  - 2023-12-31  \
Datum                                                       
2024-01-01 00:00:00+01:00                            8.79   
2024-01-01 01:00:00+01:00                            8.71   
2024-01-01 02:00:00+01:00                            7.95   
2024-01-01 03:00:00+01:00                            5.28   
2024-01-01 04:00:00+01:00                            5.84   

                           T(°C) - 2023-01-01  - 2023-12-31  
Datum        

In [12]:
TILT_DEG    = 25.0
AZIMUTH_DEG = 135.0   # 0=N, 90=E, 180=S, 270=W → SE = 135°
ALBEDO      = 0.2
K_ILL       = 120.0

In [13]:
solpos = solarposition.get_solarposition(
    time=datum, latitude=latitude, longitude=longitude, altitude=elevation
)

In [14]:
poa = irradiance.get_total_irradiance(
    surface_tilt=TILT_DEG,
    surface_azimuth=AZIMUTH_DEG,
    dni=clearsky["dni"],
    ghi=clearsky["ghi"],
    dhi=clearsky["dhi"],
    solar_zenith=solpos["apparent_zenith"],
    solar_azimuth=solpos["azimuth"],
    albedo=ALBEDO,
)

In [15]:
poa_global = poa["poa_global"]

In [16]:
poa_global

Datum
2024-01-01 00:00:00+01:00    0.0
2024-01-01 01:00:00+01:00    0.0
2024-01-01 02:00:00+01:00    0.0
2024-01-01 03:00:00+01:00    0.0
2024-01-01 04:00:00+01:00    0.0
                            ... 
2024-12-31 19:00:00+01:00    0.0
2024-12-31 20:00:00+01:00    0.0
2024-12-31 21:00:00+01:00    0.0
2024-12-31 22:00:00+01:00    0.0
2024-12-31 23:00:00+01:00    0.0
Name: poa_global, Length: 8783, dtype: float64

In [17]:
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "plotly_mimetype"  # Or PLOTLY_RENDERER=plotly_mimetype

# Assume df is your DataFrame with a tz-aware DatetimeIndex and one or more value columns
x = poa_global.index.tz_convert(TZ).tz_localize(None)
df = poa_global.to_frame(name='poa_global')

fig = px.line(
    df.assign(__x__=x),                 # keep columns, add naive datetime
    x="__x__",                          # x-axis
    y=[c for c in df.columns],          # all numeric columns; or y="y" for single series
    title="Total Clear Sky Irradiance at Solbyn, Dalby, Sweden (2024)",
)
fig.update_layout(
    hovermode="x unified",
    xaxis_title="Date",
    yaxis_title="Irradiance (W/m²)",
    xaxis=dict(rangeslider=dict(visible=True)),  # drag to zoom with slider
)
fig.show()
#fig.write_html("plots/clearsky.html")

In [18]:
#%whos

In [19]:
# Daily energy + weekly scan (uses existing systems, pd, go, TZ)
import plotly.graph_objects as go

# 1) Index → local-naive
t = pd.to_datetime(systems.index)
if getattr(t, "tz", None) is not None:
    t = t.tz_convert(TZ).tz_localize(None)
sys_ = systems.copy()
sys_.index = t

# 2) Column bindings (edit if different)
prod_col = "Produktion"          # PV production (assumed kWh per interval)
grid_in_col = "El kWh"           # Grid import (kWh per interval)
temp_col = "Utomhustemperatur"   # Ambient °C (optional)

# 3) Daily aggregates (kWh/day, °C/day)
prod_kwh_d     = sys_[prod_col].resample("D").sum().rename("PV (kWh)")
grid_in_kwh_d  = sys_[grid_in_col].resample("D").sum().rename("Grid in (kWh)")
temp_c_d       = (sys_[temp_col].resample("D").mean().rename("Ambient (°C)")
                  if temp_col in sys_.columns else None)
net_kwh_d      = (prod_kwh_d - grid_in_kwh_d).rename("Net (kWh)")  # proxy: export minus import (may differ from true feed-in)

# 4) Plot: bars (prod/import), net line, temp on y2; weekly range buttons
fig_weekly = go.Figure()
fig_weekly.add_bar(x=prod_kwh_d.index,    y=prod_kwh_d.values,    name=prod_kwh_d.name)
fig_weekly.add_bar(x=grid_in_kwh_d.index, y=-grid_in_kwh_d.values, name="Grid in (kWh)")  # negative to stack below zero
fig_weekly.add_scatter(x=net_kwh_d.index, y=net_kwh_d.values, name=net_kwh_d.name, mode="lines")
if temp_c_d is not None:
    fig_weekly.add_scatter(x=temp_c_d.index, y=temp_c_d.values, name=temp_c_d.name, mode="lines", yaxis="y2")

fig_weekly.update_layout(
    title="Daily energy transfer (PV vs Grid In) — weekly scan",
    barmode="relative", hovermode="x unified",
    xaxis=dict(
        title="Date",
        rangeselector=dict(buttons=[
            dict(count=7,  step="day",   label="1w", stepmode="backward"),
            dict(count=14, step="day",   label="2w", stepmode="backward"),
            dict(count=1,  step="month", label="1m", stepmode="backward"),
            dict(step="all")
        ]),
        rangeslider=dict(visible=False),
    ),
    yaxis=dict(title="Energy (kWh)"),
    yaxis2=(dict(title="Ambient (°C)", overlaying="y", side="right") if temp_c_d is not None else None),
)
fig_weekly.show()
#fig_weekly.write_html("plots/weekly_scan.html")

In [20]:
# Uses existing: pd, np, go, TZ, systems

# --- 0) Local-naive index
t = pd.to_datetime(systems.index)
if getattr(t, "tz", None) is not None:
    t = t.tz_convert(TZ).tz_localize(None)
sys_ = systems.copy(); sys_.index = t

# --- 1) Net energy per sample (kWh). Assumes columns are kWh per interval.
prod_col, grid_in_col = "Produktion", "El kWh"
net_e = (sys_[prod_col] - sys_[grid_in_col]).rename("net_kWh")

# --- 2) DAILY Net (kWh/day) — weekly/daily scanning
net_kwh_d = net_e.resample("D").sum()

# color by sign
colors = np.where(net_kwh_d.values >= 0, "rgba(33,150,243,0.7)", "rgba(244,67,54,0.7)")

fig_week_net = go.Figure()
fig_week_net.add_bar(x=net_kwh_d.index, y=net_kwh_d.values, name="Net (kWh/day)", marker_color=colors)
fig_week_net.update_layout(
    title="Daily Net Energy (kWh) — use 1w/2w to scan weeks",
    xaxis=dict(
        title="Date",
        rangeselector=dict(buttons=[
            dict(count=7,  step="day",   label="1w", stepmode="backward"),
            dict(count=14, step="day",   label="2w", stepmode="backward"),
            dict(count=1,  step="month", label="1m", stepmode="backward"),
            dict(step="all")
        ]),
        rangeslider=dict(visible=False),
    ),
    yaxis=dict(title="kWh/day"),
    hovermode="x unified",
)
fig_week_net.show()
#fig_week_net.write_html("plots/weekly_net.html")
# --- 3) TYPICAL 24h Net profile (kWh per hour-of-day)
# If your samples aren't hourly, this still averages per clock hour across days.
net_kwh_by_hod = net_e.groupby(net_e.index.hour).mean().reindex(range(24), fill_value=0.0)
hod = np.arange(24)

# --- 4) Simple same-day storage model on the typical day
cap_kwh = 30.0        # capacity to test (edit)
eta_rt  = 0.90        # round-trip efficiency (edit)
soc0    = 0.5 * cap_kwh

soc = np.zeros(24); soc[0] = soc0
residual = np.zeros(24)   # after storage (positive=export, negative=import)
soc_now = soc0

for h in range(24):
    e = net_kwh_by_hod.iloc[h]  # + = surplus, - = deficit
    if e >= 0:
        # charge with efficiency on the way in
        charge = min(e * eta_rt, cap_kwh - soc_now)
        spill = e * eta_rt - charge
        soc_now += charge
        residual[h] = spill / eta_rt  # show spill as remaining export (undo efficiency for display)
    else:
        # discharge with efficiency on the way out
        need = -e
        discharge = min(need / eta_rt, soc_now)
        soc_now -= discharge
        unmet = need - discharge * eta_rt
        residual[h] = -(unmet)        # negative import remaining
    soc[h] = soc_now

# Optional: make SOC end roughly where it started (cyclic day)
# This keeps plots comparable; skip if you want raw outcome.
soc_shift = soc_now - soc0
if abs(soc_shift) > 1e-6:
    soc -= soc_shift
    soc_now -= soc_shift

# --- 5) Plot typical-day bars (original vs after-storage) + SOC
fig_daily = go.Figure()
fig_daily.add_bar(x=hod, y=net_kwh_by_hod.values, name="Net (kWh/h)")
fig_daily.add_bar(x=hod, y=residual, name="After storage (kWh/h)")
fig_daily.add_scatter(x=hod, y=soc, name="SOC (kWh)", mode="lines", yaxis="y2")

fig_daily.update_layout(
    title="Typical 24h Net profile — storage shift test",
    barmode="relative",
    xaxis=dict(title="Hour of day (local)"),
    yaxis=dict(title="kWh per hour"),
    yaxis2=dict(title="Battery SOC (kWh)", overlaying="y", side="right"),
    hovermode="x unified",
)
fig_daily.show()
#fig_daily.write_html("plots/daily_storage_test.html")

In [21]:
.087*92000

8003.999999999999